# Atlassian Principal — ML Coding Revision
## Weighted sampling · Coupon recommendation · NumPy scoring · Tests

**One notebook. Two main problems. A script-style and a small OOP implementation of each.**
The related cosine-ranking extension also has both implementations. Talking points are in their own Markdown cells.

### Your 90-minute route

| Time | Read and run | What to reproduce from a blank cell |
|---|---|---|
| **0–30 min** | **Section 1:** intervals → script → class → boundary tests | Cumulative weights, one draw, and a test at a boundary. |
| **30–65 min** | **Section 2:** data → scoring → script → class → tests | Filter → score → deterministic top-K. |
| **65–90 min** | **Section 3**, then **Section 4** | Explain array shapes, run tests, and change a weight or scoring function. |

**Read the first implementation deeply; skim the second for what changes when state becomes reusable.**
Sections marked **Optional reference** are there for follow-up revision; do not memorize everything in one sitting.

### Run instructions

Use **Python 3.10+**. The two main problems need only the standard library. Section 3 additionally needs **NumPy**.
Run cells from top to bottom, or use **Restart Kernel → Run All**. All data are included. There are no API calls, downloads, or external datasets.
The saved notebook includes executed examples and test results.

### Scope and provenance

The topics follow the preceding 90-minute preparation plan. The detailed prompts, coupon rules, toy data, scoring formula, code, and tests below are **original practice exercises**, not verified verbatim Atlassian interview questions. API references are listed at the end and cited by number in the explanations.

## Contents

1. [Weighted sampling](#weighted-sampling) — cumulative weights, binary search, RNG injection, safe updates.
2. [Coupon recommendation](#coupon-recommendation) — history, eligibility, scoring, top-K, cold start.
3. [Cosine ranking extension](#cosine-ranking) — NumPy shapes, normalization, masks, cached item vectors.
4. [Testing and change-request rehearsal](#testing-rehearsal) — plain assertions and `unittest`.
5. [Optional top-K reference and final recall](#top-k-reference).
6. [Official API references](#references).

**Conventions:** a leading `_` means “internal helper” by convention; `k` is a nonnegative Python integer; IDs are unique strings where ranking needs an ID tie-break. Code uses small functions and classes, not a production framework.

In [1]:
# Standard-library setup: enough for Sections 1 and 2.
import sys
import math
import random
import heapq
import unittest
from bisect import bisect_left, bisect_right
from collections import Counter
from copy import deepcopy
from datetime import date
from itertools import accumulate

print("Python:", sys.version.split()[0])
print("Standard-library setup ready. NumPy is imported in Section 3.")

Python: 3.13.5
Standard-library setup ready. NumPy is imported in Section 3.


### Function vs. object: the mental model

| Term | Meaning in this notebook |
|---|---|
| **Function** | Inputs arrive as arguments; the function performs work and returns a result. |
| **Class** | A definition of an object that keeps related data and operations together. |
| `__init__` | Initialization code run when you construct an instance, e.g. `WeightedSampler(...)`. |
| `self` | The particular instance receiving a method call. `sampler.sample()` supplies it automatically. |
| `self._cumulative` | Data retained between calls, instead of being rebuilt inside every function call. |
| **Method** | A function attached to a class, such as `sample_many`. |
| **Callback** | A function passed as a value, such as a replaceable coupon-scoring function. |
| `rng=None` | Create a generator inside the call/constructor when none is supplied; avoid a shared mutable default. |

For this revision, **script-style** means ordinary functions plus a visible driver cell. **OOP** means a small state-owning class; there is no interface hierarchy or factory machinery. [R11]

In [2]:
def validate_k(k):
    """Our contract: k must be a nonnegative Python int; bool does not count."""
    if type(k) is not int or k < 0:
        raise ValueError("k must be a nonnegative Python integer")


def expect_raises(error_type, operation):
    """Tiny test helper: operation is a zero-argument function to execute.

    Example: expect_raises(ValueError, lambda: validate_k(-1))
    The lambda delays execution until this helper can catch the exception.
    """
    try:
        operation()
    except error_type:
        return
    raise AssertionError(f"Expected {error_type.__name__} to be raised")


validate_k(0)
expect_raises(ValueError, lambda: validate_k(-1))
print("Validation and test helpers ready.")

Validation and test helpers ready.


<a id="weighted-sampling"></a>
# 1. Weighted sampling — highest priority

### Practice problem

Given `items` and nonnegative `weights` of equal length, draw items **with replacement** so that

\[
P(\text{item at index }i)=\frac{w_i}{\sum_j w_j}.
\]

| Input / output | Contract |
|---|---|
| `items` | Nonempty sequence. We sample **positions**; repeated labels are allowed and their probabilities add. |
| `weights` | Ordinary Python `int` or `float` values, nonnegative and finite; at least one is positive. They need not sum to 1. |
| `k` | Number of independent draws; `0` returns an empty list for a valid distribution. |
| `rng` | A random-number generator object. Supplying it makes examples reproducible and tests controllable. |
| Result | The function returns a **list of items**, even when `k=1`. The class also exposes a one-item `sample()` method. |

**Ask before coding:** Are weights integers or floats? Static or updated? Replacement or no replacement? Return IDs or indexes? One draw or many?

We support integer and floating-point weights with the same cumulative-weight representation. Integer weights use an integer draw; float weights use a real-valued draw. [R1]

### Intuition: each weight owns an interval

```text
items:       A   B   C   D
weights:     1   3   0   2
cumulative:  1   4   4   6
```

| Item | Interval | Integer tickets in it | Probability |
|---|---|---|---|
| A | `[0, 1)` | `0` | `1/6` |
| B | `[1, 4)` | `1, 2, 3` | `3/6` |
| C | `[4, 4)` | None | `0` |
| D | `[4, 6)` | `4, 5` | `2/6` |

**Algorithm:** draw a ticket in `[0, total)` and find the first cumulative value **strictly greater** than it.

A ticket of `4` belongs to **D**. `bisect_right([1, 4, 4, 6], 4)` returns index `3`, skipping both cumulative `4`s. `bisect_left` would give the wrong bucket for this interval convention. [R2]

**Why it works:** the interval for position `i` has length `w_i`. Uniform sampling across total length `sum(weights)` allocates the required probability to that position. For integer weights, this is exactly a count of equally likely tickets.

In [3]:
# Walk the algorithm before wrapping it in functions.
items_demo = ["A", "B", "C", "D"]
weights_demo = [1, 3, 0, 2]

running_total = 0
cumulative_demo = []
for weight in weights_demo:
    running_total += weight
    cumulative_demo.append(running_total)

# accumulate produces running sums lazily; list materializes the results.
assert cumulative_demo == list(accumulate(weights_demo))
print("Cumulative weights:", cumulative_demo)

for ticket in range(cumulative_demo[-1]):
    index = bisect_right(cumulative_demo, ticket)
    print(f"ticket={ticket} -> index={index} -> item={items_demo[index]}")

print("At ticket 4: bisect_left =", bisect_left(cumulative_demo, 4),
      "; bisect_right =", bisect_right(cumulative_demo, 4))

Cumulative weights: [1, 4, 4, 6]
ticket=0 -> index=0 -> item=A
ticket=1 -> index=1 -> item=B
ticket=2 -> index=1 -> item=B
ticket=3 -> index=1 -> item=B
ticket=4 -> index=3 -> item=D
ticket=5 -> index=3 -> item=D
At ticket 4: bisect_left = 1 ; bisect_right = 3


### API intuition — the parameters you should recognize

| Expression | Read it as |
|---|---|
| `list(accumulate(weights))` | “Store each running sum.” The input is the sequence of weights. |
| `cumulative[-1]` | “The last running sum,” hence the total weight. |
| `rng.randrange(total)` | One integer from `0` through `total - 1`; the upper bound is excluded. |
| `rng.random()` | One floating-point value in `[0, 1)`. Multiply by total to choose a position on the weight interval. |
| `bisect_right(cumulative, target)` | The insertion index **after** any entries equal to target: the first strictly greater entry. |
| `random.Random(7)` | A local generator initialized with seed 7. Keep using this object so its state advances. |
| `for _ in range(k)` | Repeat `k` times; `_` signals that the loop counter is unused. |

Do not rebuild or reseed a generator for every draw. Two newly created generators with the same seed reproduce a sequence; successive calls to **one** generator advance through it. [R1, R2, R10]

In [4]:
def make_cumulative(weights):
    """Validate a distribution and return cumulative weights.

    This shared helper keeps the script and class on the same contract.
    Validation is O(n); constructing the running sums is also O(n).
    """
    if len(weights) == 0:
        raise ValueError("weights must not be empty")

    for weight in weights:
        # This intentionally accepts ordinary Python int/float values only.
        # Checking type exactly also rejects True/False as weights.
        if type(weight) not in (int, float):
            raise ValueError("weights must be Python ints or floats")
        if weight < 0:
            raise ValueError("weights must be nonnegative")
        if isinstance(weight, float) and not math.isfinite(weight):
            raise ValueError("weights must be finite")

    cumulative = list(accumulate(weights))
    total = cumulative[-1]
    if total <= 0:
        raise ValueError("at least one weight must be positive")
    if isinstance(total, float) and not math.isfinite(total):
        raise ValueError("total weight must be finite")
    return cumulative


def draw_weighted_index(cumulative, rng):
    """One O(log n) draw from an already validated cumulative array."""
    total = cumulative[-1]

    if isinstance(total, int):
        # Exact integer-ticket version: no conversion to float is required.
        target = rng.randrange(total)
    else:
        target = rng.random() * total
        # Small numerical guard: multiplication may round up to total.
        # nextafter(total, 0.0) is the closest representable float below total.
        # Keeping target < total also avoids selecting a trailing zero weight.
        target = min(target, math.nextafter(total, 0.0))

    return bisect_right(cumulative, target)

### Numerical note — understand, do not overengineer

The integer branch avoids floating-point interval rounding. The float branch is the usual finite-precision approximation: extremely tiny weights next to huge ones can disappear in cumulative addition. The `nextafter` guard protects the **upper boundary**, not all possible precision issues. In an interview, state that limitation rather than silently promising exact probabilities for arbitrary real numbers.

**No normalization step is needed:** `[1, 3, 0, 2]` and `[10, 30, 0, 20]` define the same probabilities.

## 1A. Script-style implementation

State is local to one call. For a batch of `k` draws, we build the cumulative array **once**, not once per draw.
The only shared dependencies are the two short helpers directly above.

In [5]:
def weighted_sample(items, weights, k=1, rng=None):
    """Return k weighted draws with replacement.

    items, weights : equal-length sequences
    k              : how many items to return, not a probability or top-K rank
    rng            : optional local RNG; caller can supply random.Random(seed)

    Time: O(n + k log n). Extra space: O(n + k), including the output list.
    """
    validate_k(k)
    if len(items) != len(weights):
        raise ValueError("items and weights must have the same length")

    cumulative = make_cumulative(weights)
    if rng is None:
        rng = random.Random()

    chosen = []
    for _ in range(k):
        index = draw_weighted_index(cumulative, rng)
        chosen.append(items[index])
    return chosen


# Driver: named parameters make the call easy to read during an interview.
print("One draw:", weighted_sample(items_demo, weights_demo,
                                   k=1, rng=random.Random(7)))
print("Batch:", weighted_sample(items_demo, weights_demo,
                                k=12, rng=random.Random(7)))
print("Float weights:", weighted_sample(["low", "high"], [0.2, 0.8],
                                        k=8, rng=random.Random(7)))

One draw: ['B']
Batch: ['B', 'B', 'B', 'D', 'A', 'A', 'D', 'A', 'B', 'D', 'A', 'D']
Float weights: ['high', 'low', 'high', 'low', 'high', 'high', 'low', 'high']


## 1B. OOP implementation

The object owns the item list, weights, cumulative array, and RNG. Construction pays the `O(n)` setup cost once; repeated `sample()` calls reuse it.

`list(items)` and `list(weights)` copy the outer sequences, so later edits to the caller's lists do not change this sampler accidentally. Items themselves are not deep-copied. The leading `_` marks internal state by convention. [R11]

In [6]:
class WeightedSampler:
    """A reusable sampler with small, explicit state."""

    def __init__(self, items, weights, rng=None):
        if len(items) != len(weights):
            raise ValueError("items and weights must have the same length")

        # Instance attributes survive after __init__ finishes.
        self._items = list(items)
        self._weights = list(weights)
        self._cumulative = make_cumulative(self._weights)
        self._rng = random.Random() if rng is None else rng

    def sample(self):
        """Return ONE item. Setup is already done; this costs O(log n)."""
        index = draw_weighted_index(self._cumulative, self._rng)
        return self._items[index]

    def sample_many(self, k=1):
        """Return a LIST of k items, with replacement."""
        validate_k(k)
        return [self.sample() for _ in range(k)]

    def update_weight(self, index, new_weight):
        """Replace one weight and rebuild in O(n).

        Important: validate candidate state BEFORE changing current state.
        A rejected update must leave the sampler usable.
        """
        if type(index) is not int or not 0 <= index < len(self._weights):
            raise ValueError("index must identify an existing position")

        candidate_weights = self._weights.copy()
        candidate_weights[index] = new_weight
        candidate_cumulative = make_cumulative(candidate_weights)

        # Commit only after the candidate distribution has passed validation.
        self._weights = candidate_weights
        self._cumulative = candidate_cumulative


sampler_demo = WeightedSampler(items_demo, weights_demo, rng=random.Random(7))
print("Object batch:", sampler_demo.sample_many(12))
print("Next item from the SAME generator:", sampler_demo.sample())

Object batch: ['B', 'B', 'B', 'D', 'A', 'A', 'D', 'A', 'B', 'D', 'A', 'D']
Next item from the SAME generator: B


### Test the boundaries, not just the random frequencies

A seed is useful for repeatability. A **controlled RNG** is stronger for a boundary test: it lets us force exactly the tickets we care about.

The test object below implements the tiny interface the sampler uses: `randrange(stop)` and `random()`. It does not inherit from `random.Random`; Python only needs the called methods to exist. This is a concrete example of dependency injection and duck typing.

The fake controls the random **input**. The real cumulative-array and binary-search code still executes.

In [7]:
class FixedRNG:
    """Test-only source of predetermined integer tickets or float fractions."""

    def __init__(self, integers=(), fractions=()):
        self._integers = iter(integers)
        self._fractions = iter(fractions)

    def randrange(self, stop):
        value = next(self._integers)
        assert 0 <= value < stop, "Test ticket is outside the RNG contract"
        return value

    def random(self):
        value = next(self._fractions)
        assert 0 <= value < 1, "Test fraction is outside the RNG contract"
        return value


def test_sampler_contract():
    # Both entry points are adapted to the same signature for the tests.
    versions = {
        "script": lambda items, weights, k, rng: weighted_sample(items, weights, k, rng),
        "oop": lambda items, weights, k, rng: WeightedSampler(items, weights, rng).sample_many(k),
    }
    for name, draw in versions.items():
        # Exhaust ALL integer tickets: the exact bucket mapping is observable.
        assert draw(["A", "B", "C", "D"], [1, 3, 0, 2], 6,
                    FixedRNG(integers=range(6))) == ["A", "B", "B", "B", "D", "D"]

        # Float boundary test, including an internal and a trailing zero weight.
        assert draw(["A", "B", "C", "D", "E"], [0.25, 0.25, 0.0, 0.5, 0.0], 4,
                    FixedRNG(fractions=[0.0, 0.25, 0.5, math.nextafter(1.0, 0.0)])) == ["A", "B", "D", "D"]

        assert draw(["only"], [5], 3, random.Random(1)) == ["only"] * 3
        assert draw(["A"], [1], 0, random.Random(1)) == []
        assert draw(["same", "same"], [1, 2], 4, random.Random(1)) == ["same"] * 4

        bad_distributions = [([], []), (["A"], [1, 2]), (["A"], [0]),
                             (["A"], [-1]), (["A"], [float("nan")]),
                             (["A"], [float("inf")]), (["A"], [True]),
                             (["A", "B"], [1e308, 1e308])]
        for bad_items, bad_weights in bad_distributions:
            expect_raises(ValueError, lambda: draw(bad_items, bad_weights, 1, random.Random(1)))
        for bad_k in [-1, 1.5, True]:
            expect_raises(ValueError, lambda: draw(["A"], [1], bad_k, random.Random(1)))
        print(name, "sampler: boundaries and invalid-input cases passed")

    # Same algorithm + same seed + same inputs => matching implementations.
    assert weighted_sample(items_demo, weights_demo, 30, random.Random(19)) == \
        WeightedSampler(items_demo, weights_demo, random.Random(19)).sample_many(30)


test_sampler_contract()

script sampler: boundaries and invalid-input cases passed
oop sampler: boundaries and invalid-input cases passed


In [8]:
# Distribution sanity check. This is a demonstration, NOT an exact-frequency test.
# A correct sampler does not promise counts of exactly 1,000 / 3,000 / 0 / 2,000.
draw_count = 6_000
observed = Counter(weighted_sample(items_demo, weights_demo, draw_count, random.Random(42)))
for item, weight in zip(items_demo, weights_demo):
    theoretical = weight / sum(weights_demo)
    empirical = observed[item] / draw_count
    print(f"{item}: expected={theoretical:.3f}, observed={empirical:.3f}, count={observed[item]}")
assert observed["C"] == 0  # Zero probability is a deterministic invariant.

A: expected=0.167, observed=0.162, count=970
B: expected=0.500, observed=0.494, count=2964
C: expected=0.000, observed=0.000, count=0
D: expected=0.333, observed=0.344, count=2066


### Change request: “Now change one item's weight”

**Script approach:** change a copied weight list and call the function again. It rebuilds the cumulative array.

**OOP approach:** call `update_weight`; it rebuilds the stored cumulative array. This is still `O(n)` per update, deliberately simple.

Changing weight `w[i]` affects every cumulative entry from `i` onward. Updating only `cumulative[i]` is incorrect.

In [9]:
def test_weight_update():
    updated_weights = [1, 0, 0, 2]  # B is now impossible.
    script_result = weighted_sample(items_demo, updated_weights, 20, random.Random(8))

    obj = WeightedSampler(items_demo, weights_demo, random.Random(8))
    obj.update_weight(index=1, new_weight=0)
    assert obj.sample_many(20) == script_result
    assert "B" not in script_result

    # Rejecting an all-zero update must not destroy valid existing state.
    one_live_item = WeightedSampler(["A", "B"], [1, 0], random.Random(8))
    expect_raises(ValueError, lambda: one_live_item.update_weight(0, 0))
    assert one_live_item.sample_many(5) == ["A"] * 5
    expect_raises(ValueError, lambda: one_live_item.update_weight(-1, 2))

    # An int distribution can become a float distribution after a valid update.
    changing_type = WeightedSampler(["A", "B"], [1, 1], random.Random(3))
    changing_type.update_weight(0, 0.25)
    assert changing_type.sample_many(12) == weighted_sample(
        ["A", "B"], [0.25, 1], 12, random.Random(3))
    print("Weight-update checks passed.")


test_weight_update()

Weight-update checks passed.


### Talking points — weighted sampling

> “I will treat weights as relative probability mass and confirm whether sampling is with replacement.”

> “I will build cumulative weights once, then draw a position and find the first strictly greater cumulative boundary.”

> “For these half-open intervals, I need `bisect_right`; a zero weight owns an empty interval.”

> “The function costs `O(n + k log n)` per batch. The object pays `O(n)` at construction and `O(log n)` per draw; it stores `O(n)` state.”

> “I will test exact bucket boundaries using an injected RNG. A frequency check is supplementary, not proof of correctness.”

> “For updates, my first version rebuilds in `O(n)`. If frequent updates dominate at large scale, I would discuss a prefix-sum tree instead of coding one prematurely.”

**Common mistakes:** using an inclusive upper-bound ticket; reseeding on every draw; normalizing unnecessarily; selecting zero-weight items at a boundary; claiming the function's setup is free; partially modifying state before rejecting an update.

### Optional reference — sampling without replacement is a different contract

For **sequential probability-proportional-to-remaining-weight** sampling, choose one item, remove that position, rebuild the remaining distribution, and repeat. At each step:

\[
P(i\mid\text{remaining}) = \frac{w_i}{\sum_{j\in\text{remaining}}w_j}.
\]

The simple rebuild approach costs `O(k n)`. Require `k` to be no greater than the number of positive-weight positions. Clarify whether uniqueness means **positions** or **distinct IDs**.

Taking exactly `k` with-replacement draws and then applying `set(...)` may return fewer than `k` items. Resampling duplicates until you have `k` distinct positions can realize the sequential rule, but can become very slow for concentrated weights. The final inclusion probability is not generally `k * w_i / total`.

`random.choices(..., weights=..., k=...)` is a useful library equivalent for **with-replacement** use. `random.sample(..., counts=...)` treats counts as repeated population occurrences; it is not a guarantee of distinct weighted labels. [R1]

<a id="coupon-recommendation"></a>
# 2. Coupon recommendation — filter → score → top-K

### Practice problem and explicit assumptions

Implement `recommend_coupons(user, coupons, history, k, today)`.

| Input | Fields / interpretation |
|---|---|
| `user` | `id`, `cart_total` |
| Each coupon | Unique string `id`, `category`, `expires_on` as a `date`, `min_spend`, `popularity` in `[0, 1]` |
| Each history row | A completed redemption with `user_id`, `coupon_id`, `category`; rows are assumed to be unique events. |
| `today` | Explicit evaluation date, not a hidden call to the machine's clock. |
| Result | List of `(coupon_id, score)` pairs, ranked by score descending, then ID ascending. |

**Eligibility:** a coupon is valid through its expiry date, the cart meets minimum spend, and this user has not redeemed it.
**Cold start:** no redemption history for this user → rank eligible coupons by popularity.
**Duplicate catalog IDs:** reject them rather than invent a “first wins” or “last wins” rule.
**`k=0` / too few eligible coupons:** return `[]` / return all available eligible results.

These are **chosen practice rules**, not claims about an actual coupon product or an interviewer's exact requirements. We assume required fields and their basic types follow the schema; lightweight checks cover the consequential numerical and uniqueness constraints.

In [10]:
# A fixed date makes the examples repeatable next week as well as today.
TODAY = date(2026, 9, 11)
USER = {"id": "u1", "cart_total": 800.0}

COUPONS = [
    {"id": "A", "category": "books", "expires_on": date(2026, 9, 30), "min_spend": 0,    "popularity": 0.9},
    {"id": "B", "category": "books", "expires_on": date(2026, 9, 30), "min_spend": 100,  "popularity": 0.6},
    {"id": "C", "category": "music", "expires_on": date(2026, 9, 30), "min_spend": 100,  "popularity": 0.8},
    {"id": "D", "category": "travel", "expires_on": date(2026, 9, 10), "min_spend": 0,   "popularity": 1.0},
    {"id": "E", "category": "music", "expires_on": date(2026, 9, 30), "min_spend": 1000, "popularity": 1.0},
    {"id": "F", "category": "books", "expires_on": date(2026, 9, 30), "min_spend": 100,  "popularity": 0.6},
    {"id": "G", "category": "games", "expires_on": TODAY,              "min_spend": 0,    "popularity": 1.0},
]

HISTORY = [
    {"user_id": "u1", "coupon_id": "A",         "category": "books"},
    {"user_id": "u1", "coupon_id": "old_book",  "category": "books"},
    {"user_id": "u1", "coupon_id": "old_music", "category": "music"},
    {"user_id": "someone_else", "coupon_id": "G", "category": "games"},
]

print("A: already redeemed; D: expired; E: cart below minimum.")
print("Eligible for u1: B, C, F, G. G expires today and is still valid.")

A: already redeemed; D: expired; E: cart below minimum.
Eligible for u1: B, C, F, G. G expires today and is still valid.


### Scoring intuition: an explainable baseline, not a trained model

Let category affinity be the fraction of this user's redemptions in that category:

\[
\text{affinity}(c) = \frac{\#\text{user redemptions in category }c}{\#\text{all user redemptions}}.
\]

For users with history:

\[
\text{score}(coupon) = \alpha\,\text{affinity}(coupon.category)
+ (1-\alpha)\,\text{popularity}(coupon), \qquad \alpha=0.7.
\]

`alpha` is a **practice configuration choice**, not a measured optimum. Popularity is assumed to already be on `[0,1]`; arbitrary raw counts should not be mixed with affinities at this weight.

For `u1`, books affinity is `2/3`, music is `1/3`, and games is `0`:

| Coupon | Affinity | Popularity | Score at `alpha=0.7` |
|---|---:|---:|---:|
| B | 2/3 | 0.6 | 0.6467 |
| F | 2/3 | 0.6 | 0.6467 |
| C | 1/3 | 0.8 | 0.4733 |
| G | 0 | 1.0 | 0.3000 |

So the expected order is **B, F, C, G**. B wins the exact score tie with F because its ID sorts first.

In [11]:
def validate_alpha(alpha):
    if not math.isfinite(alpha) or not 0 <= alpha <= 1:
        raise ValueError("alpha must be finite and between 0 and 1")


def validate_catalog(coupons):
    """Check the few schema rules that directly affect filtering and ranking."""
    ids = [coupon["id"] for coupon in coupons]
    if any(not isinstance(item_id, str) for item_id in ids):
        raise ValueError("coupon IDs must be strings")
    if len(ids) != len(set(ids)):
        raise ValueError("coupon IDs must be unique")
    for coupon in coupons:
        popularity = coupon["popularity"]
        minimum = coupon["min_spend"]
        if not math.isfinite(popularity) or not 0 <= popularity <= 1:
            raise ValueError("popularity must be finite and in [0, 1]")
        if not math.isfinite(minimum) or minimum < 0:
            raise ValueError("minimum spend must be finite and nonnegative")


def build_user_context(user_id, history):
    """Convert this user's redemption rows into affinities and a redeemed-ID set.

    We scan history once: O(H) time. Other users' records are ignored.
    """
    category_counts = Counter()  # Missing categories behave like a count of 0.
    redeemed_ids = set()         # Membership checks avoid repeatedly scanning history.

    for event in history:
        if event["user_id"] != user_id:
            continue
        category_counts[event["category"]] += 1
        redeemed_ids.add(event["coupon_id"])

    total_redemptions = sum(category_counts.values())
    affinities = {}
    if total_redemptions > 0:
        for category, count in category_counts.items():
            affinities[category] = count / total_redemptions
    return affinities, redeemed_ids


def baseline_coupon_score(coupon, affinities, alpha=0.7):
    """Score one already-eligible coupon. Bigger is better, not a probability."""
    if not affinities:
        # Cold start gets the full popularity score, not a scaled-down version.
        return coupon["popularity"]
    preference = affinities.get(coupon["category"], 0.0)
    return alpha * preference + (1 - alpha) * coupon["popularity"]


affinities_demo, redeemed_demo = build_user_context(USER["id"], HISTORY)
print("Affinities:", affinities_demo)
print("Redeemed IDs:", sorted(redeemed_demo))

Affinities: {'books': 0.6666666666666666, 'music': 0.3333333333333333}
Redeemed IDs: ['A', 'old_book', 'old_music']


### Python functions/methods used here

| Expression | Intuition |
|---|---|
| `Counter()` / `counts[category] += 1` | Maintain a count per category; missing keys start from zero. |
| `event["user_id"]` | Required-field lookup; a missing field is a schema error. |
| `affinities.get(category, 0.0)` | Optional lookup; an unseen category gets the explicit default. |
| `redeemed_ids.add(id)` / `id in redeemed_ids` | Insert into a set / test membership. |
| `counts.items()` | Iterate `(key, value)` pairs. |
| `sorted(rows, key=lambda row: (-row[1], row[0]))` | Sort by negative score, then ID. Ordinary ascending tuple order now means score descending, ID ascending. |
| `rows[:k]` | Take at most the first `k` rows; oversized `k` is safe. |

`sorted(...)` returns a new list. `list.sort(...)` sorts a list in place and returns `None`. Reversing the entire `(score, id)` key would reverse ID tie-breaking too. [R3, R4]

## 2A. Script-style implementation

Start with the visible loop. Reject ineligible coupons **before** scoring and selecting the top K. The optional `scorer` callback gives a follow-up a small place to attach without changing filtering or ranking.

In [12]:
def recommend_coupons(user, coupons, history, k, today, alpha=0.7, scorer=None):
    """Return up to k (coupon_id, score) pairs.

    today  : explicit date; expires_on == today is eligible
    alpha  : balance used only by the default scorer
    scorer : optional function with signature scorer(coupon, affinities) -> number

    Time: O(H + C + M log M), with H history rows, C coupons, M eligible coupons.
    """
    validate_k(k)
    validate_alpha(alpha)
    validate_catalog(coupons)
    if k == 0:
        return []

    affinities, redeemed_ids = build_user_context(user["id"], history)
    scored = []

    for coupon in coupons:
        # 1. FILTER: each rule is independently readable and testable.
        if coupon["expires_on"] < today:
            continue
        if coupon["min_spend"] > user["cart_total"]:
            continue
        if coupon["id"] in redeemed_ids:
            continue

        # 2. SCORE: the default is explicit; a callback can replace it.
        if scorer is None:
            score = baseline_coupon_score(coupon, affinities, alpha)
        else:
            score = scorer(coupon, affinities)
        if not math.isfinite(score):
            raise ValueError("scorer must return a finite number")
        scored.append((coupon["id"], float(score)))

    # 3. SELECT: negative score gives descending order; ID breaks ties ascending.
    ranked = sorted(scored, key=lambda row: (-row[1], row[0]))
    return ranked[:k]


script_recommendations = recommend_coupons(USER, COUPONS, HISTORY, k=3, today=TODAY)
for coupon_id, score in script_recommendations:
    print(f"{coupon_id}: {score:.4f}")

B: 0.6467
F: 0.6467
C: 0.4733


## 2B. OOP implementation

The object owns a catalog snapshot and a scoring policy. **The user, history, and evaluation date remain request inputs**, so reusing one object across users does not reuse the previous user's redemption context.

The class does not wrap `recommend_coupons`; its method explicitly performs the pipeline. Both versions share only the small validation/context/scoring helpers so they agree on the contract.

In [13]:
class CouponRecommender:
    """Reusable catalog + policy; per-user context is rebuilt per request."""

    def __init__(self, coupons, alpha=0.7, scorer=None):
        validate_alpha(alpha)
        validate_catalog(coupons)
        # Coupon values here are strings, numbers, and immutable date objects.
        # A shallow copy of EACH dictionary is enough for this flat schema.
        self._coupons = [coupon.copy() for coupon in coupons]
        self._alpha = alpha
        self._scorer = scorer

    def is_eligible(self, coupon, user, redeemed_ids, today):
        """A boolean predicate: no scoring or state changes here."""
        return (
            coupon["expires_on"] >= today
            and coupon["min_spend"] <= user["cart_total"]
            and coupon["id"] not in redeemed_ids
        )

    def score(self, coupon, affinities):
        """One replaceable policy, isolated from eligibility and top-K."""
        if self._scorer is None:
            value = baseline_coupon_score(coupon, affinities, self._alpha)
        else:
            value = self._scorer(coupon, affinities)
        if not math.isfinite(value):
            raise ValueError("scorer must return a finite number")
        return float(value)

    def recommend(self, user, history, k, today):
        validate_k(k)
        if k == 0:
            return []
        affinities, redeemed_ids = build_user_context(user["id"], history)
        scored = []
        for coupon in self._coupons:
            if self.is_eligible(coupon, user, redeemed_ids, today):
                scored.append((coupon["id"], self.score(coupon, affinities)))
        return sorted(scored, key=lambda row: (-row[1], row[0]))[:k]


recommender_demo = CouponRecommender(COUPONS, alpha=0.7)
object_recommendations = recommender_demo.recommend(USER, HISTORY, k=3, today=TODAY)
assert object_recommendations == script_recommendations
print("Object:", [(item_id, round(score, 4)) for item_id, score in object_recommendations])

Object: [('B', 0.6467), ('F', 0.6467), ('C', 0.4733)]


### Tests to run before optimizing

| Case | Expected behavior |
|---|---|
| Current example | B, F, C, G; A/D/E never appear. |
| Exact score tie | Smaller string ID comes first, regardless of input order. |
| Expiry equals today / spend equals minimum | Eligible. |
| No user history | Popularity fallback, still subject to eligibility. |
| Another user redeemed a coupon | Does not exclude it for the current user. |
| `k=0`, `k` too large, no eligible items | Empty / fewer than K / empty result. |
| Duplicate IDs, invalid alpha, non-finite score | Reject with a clear exception. |

The expected order below is derived from the hand-worked example, not merely from agreement between two implementations.

In [14]:
def coupon_versions():
    # Both adapters expose the SAME callable signature to the test loop.
    return {
        "script": recommend_coupons,
        "oop": lambda user, coupons, history, k, today, alpha=0.7, scorer=None:
            CouponRecommender(coupons, alpha, scorer).recommend(user, history, k, today),
    }


def test_coupon_contract():
    for name, recommend in coupon_versions().items():
        result = recommend(USER, COUPONS, HISTORY, 10, TODAY)
        assert [item_id for item_id, _ in result] == ["B", "F", "C", "G"]
        assert math.isclose(result[0][1], 0.7 * (2 / 3) + 0.3 * 0.6)
        assert recommend(USER, COUPONS, HISTORY, 0, TODAY) == []
        assert recommend(USER, [], HISTORY, 3, TODAY) == []
        assert recommend(USER, COUPONS, HISTORY, 3, date(2027, 1, 1)) == []

        # Reordering input cannot change the specified tie-break.
        reversed_result = recommend(USER, list(reversed(COUPONS)), HISTORY, 10, TODAY)
        assert reversed_result == result

        # Unknown user has no affinities or redeemed IDs; popularity wins.
        new_user = {"id": "new", "cart_total": 800.0}
        cold = recommend(new_user, COUPONS, HISTORY, 10, TODAY)
        assert [item_id for item_id, _ in cold] == ["G", "A", "C", "B", "F"]
        assert cold[0] == ("G", 1.0)

        # Exact boundary: this coupon expires today and requires exactly the cart value.
        boundary = [{"id": "Z", "category": "books", "expires_on": TODAY,
                     "min_spend": 800.0, "popularity": 0.5}]
        assert recommend(new_user, boundary, [], 1, TODAY) == [("Z", 0.5)]
        print(name, "coupon recommender: ranking, eligibility, and cold-start cases passed")

    # One shared object can safely process different users in sequence.
    shared = CouponRecommender(COUPONS)
    shared.recommend(USER, HISTORY, 3, TODAY)
    assert shared.recommend({"id": "new", "cart_total": 800}, HISTORY, 1, TODAY) == [("G", 1.0)]


def test_coupon_errors_and_input_safety():
    for name, recommend in coupon_versions().items():
        expect_raises(ValueError, lambda: recommend(USER, COUPONS + [COUPONS[0]], HISTORY, 3, TODAY))
        expect_raises(ValueError, lambda: recommend(USER, COUPONS, HISTORY, -1, TODAY))
        expect_raises(ValueError, lambda: recommend(USER, COUPONS, HISTORY, 3, TODAY, alpha=1.2))
        expect_raises(ValueError, lambda: recommend(
            USER, COUPONS, HISTORY, 3, TODAY, scorer=lambda coupon, affinities: float("nan")))

        before = deepcopy(COUPONS)
        recommend(USER, COUPONS, HISTORY, 3, TODAY)
        assert COUPONS == before  # Ranking must not mutate the caller's catalog.
    print("Coupon error and input-safety checks passed.")


test_coupon_contract()
test_coupon_errors_and_input_safety()

script coupon recommender: ranking, eligibility, and cold-start cases passed
oop coupon recommender: ranking, eligibility, and cold-start cases passed
Coupon error and input-safety checks passed.


### Talking points — coupon recommendation

> “I will clarify what makes a coupon eligible and define a deterministic tie-break before choosing a model.”

> “My baseline is filter, score, then top-K. Filtering after taking K could leave fewer results even when other eligible coupons exist.”

> “I will derive category preferences from this user's redemption history and use popularity for cold start. This score is a ranking heuristic, not a calibrated probability.”

> “For `H` history records, `C` coupons, and `M` eligible candidates, this implementation costs `O(H + C + M log M)` with a constant-time scorer. The script has `O(H + C)` worst-case extra space, including catalog-ID validation and scored rows. The object keeps an `O(C)` catalog snapshot and uses `O(H + M)` additional per-request state.”

> “I will run tests for expiry and spend boundaries, already-redeemed items, cold start, exact ties, and K larger than the eligible set.”

> “The first optimization depends on the bottleneck: index history by user, reduce candidates, or use a heap when K is much smaller than M. A learned or vector scorer can replace the heuristic without changing eligibility.”

**Principal-level boundary:** improving the model and improving the serving algorithm are different changes. Ask what inputs and objective are given; do not replace a coding task with a system-design monologue.

### Change request: “Use popularity only, but keep all eligibility rules”

Pass a different scoring function to either implementation. The callback signature stays `scorer(coupon, affinities)`. This version does not need affinities, but accepts the parameter to preserve the interface.

Expected new order for `u1`: **G, C, B, F**. A, D, and E must remain excluded.

In [15]:
def popularity_only(coupon, affinities):
    return coupon["popularity"]


def test_scorer_change():
    for name, recommend in coupon_versions().items():
        changed = recommend(USER, COUPONS, HISTORY, 10, TODAY, scorer=popularity_only)
        assert [item_id for item_id, _ in changed] == ["G", "C", "B", "F"]
        assert not ({"A", "D", "E"} & {item_id for item_id, _ in changed})
    print("Scorer replacement preserved eligibility in both implementations.")


test_scorer_change()

Scorer replacement preserved eligibility in both implementations.


<a id="cosine-ranking"></a>
# 3. Related extension — cosine similarity and top-K

This is the vector-scoring variant of the recommendation exercise, not a new claimed interview question. There are again **two implementations**: a function and a class with cached normalized item vectors.

### Practice problem

Given unique string item IDs, an item-vector matrix, one user vector, `k`, and an optional eligibility mask, return the highest-cosine eligible items. Break exact ties by ID ascending.

| Symbol / input | Shape | Meaning |
|---|---|---|
| `item_vectors` / `X` | `(N, D)` | N rows, one D-dimensional vector per item. |
| `user_vector` / `u` | `(D,)` | One query or user vector. |
| `row_norms` | `(N, 1)` | One length per item, retaining a column axis for broadcasting. |
| `scores` | `(N,)` | One scalar per item. |
| `eligible_mask` | `(N,)`, boolean | `True` means the corresponding item may be returned. |

\[
\cos(u,x_i)=\frac{u^\top x_i}{\|u\|_2\|x_i\|_2}.
\]

**Intuition:** dot product mixes direction and magnitude. Cosine compares directions after length normalization. For example, `[1,1]` and `[10,10]` have identical cosine similarity to a given nonzero query.

**Chosen zero-vector policy:** score a zero item vector as `0`; a zero user vector produces all-zero scores. The mathematical cosine is undefined there; zero is our explicit software convention. At the product layer, an unknown/zero user vector could instead trigger the popularity fallback. A zero item score can outrank a negative score; mark missing/invalid embeddings ineligible if that is undesirable.

The numerical code assumes ordinary embedding magnitudes. It is not intended as an arbitrary-extreme-value numerical library.

In [16]:
# NumPy is the only non-standard-library dependency in this notebook.
# In a notebook without NumPy, run this in a separate cell once: %pip install numpy
import numpy as np
print("NumPy:", np.__version__)

# Start with a hand-checkable example.
X_demo = np.asarray([[1, 0], [0, 1], [1, 1], [0, 0]], dtype=float)
u_demo = np.asarray([1, 0], dtype=float)

row_norms_demo = np.linalg.norm(X_demo, axis=1, keepdims=True)
unit_rows_demo = np.divide(
    X_demo,
    row_norms_demo,
    out=np.zeros_like(X_demo),
    where=row_norms_demo != 0,
)
unit_user_demo = u_demo / np.linalg.norm(u_demo)
scores_demo = unit_rows_demo @ unit_user_demo

print("X shape:", X_demo.shape)
print("row norms shape:", row_norms_demo.shape)
print("row norms:", row_norms_demo.ravel())
print("normalized rows:\n", unit_rows_demo)
print("scores shape:", scores_demo.shape)
print("scores:", np.round(scores_demo, 4))

NumPy: 2.3.5
X shape: (4, 2)
row norms shape: (4, 1)
row norms: [1.         1.         1.41421356 0.        ]
normalized rows:
 [[1.         0.        ]
 [0.         1.        ]
 [0.70710678 0.70710678]
 [0.         0.        ]]
scores shape: (4,)
scores: [1.     0.     0.7071 0.    ]


### NumPy parameter intuition — read this before the implementation

| Expression | What the argument changes |
|---|---|
| `np.asarray(values, dtype=float)` | Converts list-like data to a numeric array; `dtype=float` enables fractional values. It may reuse an existing array, so do not assume a copy. |
| `np.linalg.norm(X, axis=1)` | Collapse the feature axis of each row: one vector length per item. |
| `keepdims=True` | Keep the collapsed axis at length 1: `(N, D)` → `(N, 1)`, instead of `(N,)`. |
| `X / row_norms` | Broadcast each row's one denominator across its D features. |
| `np.divide(X, norms, out=zeros, where=norms != 0)` | Divide only where the denominator is nonzero; retain the supplied zero output elsewhere. |
| `unit_rows @ unit_user` | Matrix-vector multiplication: `(N,D) @ (D,)` → `(N,)`. Each output is one dot product. |
| `scores[i]` / `item_ids[i]` | The score and ID at the **same row index** belong together. |
| `np.allclose(actual, expected, rtol=..., atol=...)` | Compare float arrays within relative and absolute tolerances rather than exact bit equality. |

**Why `keepdims` matters:** dividing an `(N,D)` matrix by an `(N,)` array does not generally divide row-by-row. If `N == D`, it can even run while dividing the wrong axis.
**Why both `where` and `out` matter:** masked positions retain `out`; without an initialized output they need not be zero. [R5, R6, R7, R12, R13]

In [17]:
def validate_item_matrix(item_ids, item_vectors):
    """Check row/ID alignment once and return a floating-point matrix."""
    if any(not isinstance(item_id, str) for item_id in item_ids):
        raise ValueError("item IDs must be strings")
    if len(item_ids) != len(set(item_ids)):
        raise ValueError("item IDs must be unique")
    matrix = np.asarray(item_vectors, dtype=float)
    if matrix.ndim != 2 or matrix.shape[1] == 0:
        raise ValueError("item_vectors must have shape (N, D), with D > 0")
    if matrix.shape[0] != len(item_ids):
        raise ValueError("one item ID is required per matrix row")
    if not np.all(np.isfinite(matrix)):
        raise ValueError("item vectors must contain finite numbers")
    return matrix


def normalize_rows(matrix):
    """Return new unit-length rows; keep zero rows at zero."""
    row_norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    if not np.all(np.isfinite(row_norms)):
        raise ValueError("vector norms overflowed; rescale the input vectors")
    return np.divide(matrix, row_norms,
                     out=np.zeros_like(matrix), where=row_norms != 0)


def normalize_user(user_vector, dimension):
    """Validate a (D,) vector and apply our explicit zero-vector convention."""
    user = np.asarray(user_vector, dtype=float)
    if user.shape != (dimension,):
        raise ValueError(f"user_vector must have shape ({dimension},)")
    if not np.all(np.isfinite(user)):
        raise ValueError("user vector must contain finite numbers")
    length = np.linalg.norm(user)
    if not np.isfinite(length):
        raise ValueError("user-vector norm overflowed; rescale the input")
    if length == 0:
        return np.zeros_like(user)
    return user / length


def select_scored_ids(item_ids, scores, k, eligible_mask=None):
    """Apply eligibility BEFORE top-K; preserve exact-tie ordering by ID."""
    validate_k(k)
    if eligible_mask is None:
        candidate_indexes = list(range(len(item_ids)))
    else:
        mask = np.asarray(eligible_mask)
        if mask.shape != (len(item_ids),) or mask.dtype.kind != "b":
            raise ValueError("eligible_mask must be a boolean vector of length N")
        # flatnonzero gives indexes whose mask value is True.
        candidate_indexes = np.flatnonzero(mask).tolist()

    ranked_indexes = sorted(candidate_indexes,
                            key=lambda index: (-scores[index], item_ids[index]))
    return [(item_ids[index], float(scores[index])) for index in ranked_indexes[:k]]

## 3A. Script-style cosine ranker

All request inputs are visible. It normalizes the item rows on every call, then normalizes the user, computes scores, applies eligibility, and ranks.

The helpers above deliberately separate array validation/normalization from the final ranking rule. In a shorter interview answer, the same few operations can be inlined after clarifying the input assumptions.

In [18]:
def rank_cosine(item_ids, item_vectors, user_vector, k, eligible_mask=None):
    """Return up to k (item_id, cosine_score) pairs.

    item_ids       : N unique strings, aligned with the matrix rows
    item_vectors   : shape (N, D)
    user_vector    : shape (D,)
    eligible_mask  : None or N booleans; False items cannot be returned

    Time: O(ND + M log M), with M eligible items.
    """
    validate_k(k)
    matrix = validate_item_matrix(item_ids, item_vectors)
    unit_rows = normalize_rows(matrix)
    unit_user = normalize_user(user_vector, matrix.shape[1])
    scores = unit_rows @ unit_user
    return select_scored_ids(item_ids, scores, k, eligible_mask)


VECTOR_IDS = ["A", "B", "C"]
VECTOR_DATA = [[1, 0], [0, 1], [1, 1]]
MASK_A_REDEEMED = [False, True, True]

vector_script_result = rank_cosine(VECTOR_IDS, VECTOR_DATA, [1, 0], k=2,
                                   eligible_mask=MASK_A_REDEEMED)
print([(item_id, round(score, 4)) for item_id, score in vector_script_result])
# Expected: C ~0.7071, then B 0.0. A would score 1.0 but is ineligible.

[('C', 0.7071), ('B', 0.0)]


## 3B. OOP cosine ranker

The class normalizes the catalog once and stores the normalized rows. Every `rank(...)` call supplies a fresh user vector and eligibility mask.

**What caching does:** removes repeated item-normalization work. **What it does not do:** avoid scoring all N items. This is still exact full-catalog scoring, not a sublinear vector index.

In [19]:
class CosineRanker:
    """Cache fixed catalog vectors; vary the user and mask per request."""

    def __init__(self, item_ids, item_vectors):
        matrix = validate_item_matrix(item_ids, item_vectors)
        self._item_ids = list(item_ids)
        self._dimension = matrix.shape[1]
        # normalize_rows returns a new array, independent of the caller's matrix.
        self._unit_rows = normalize_rows(matrix)

    def rank(self, user_vector, k, eligible_mask=None):
        validate_k(k)
        unit_user = normalize_user(user_vector, self._dimension)
        scores = self._unit_rows @ unit_user
        return select_scored_ids(self._item_ids, scores, k, eligible_mask)


vector_ranker_demo = CosineRanker(VECTOR_IDS, VECTOR_DATA)
vector_object_result = vector_ranker_demo.rank([1, 0], k=2,
                                              eligible_mask=MASK_A_REDEEMED)
assert [x[0] for x in vector_object_result] == [x[0] for x in vector_script_result]
assert np.allclose([x[1] for x in vector_object_result], [x[1] for x in vector_script_result])
print("First user:", [(item_id, round(score, 4)) for item_id, score in vector_object_result])
print("Different user:", [(item_id, round(score, 4)) for item_id, score
                           in vector_ranker_demo.rank([0, 1], k=2)])

First user: [('C', 0.7071), ('B', 0.0)]
Different user: [('B', 1.0), ('C', 0.7071)]


### Vector tests: include a case where the wrong algorithm would look plausible

`[10,10]` has a much larger raw dot product than `[1,0]` against `[1,0]`, but **lower cosine**. Testing only already-unit vectors would not catch an accidentally unnormalized scorer.

Also test exact ties, an eligibility mask that excludes the best raw item, zero vectors, negative similarities, no eligible items, and shape errors. With negative scores, setting an ineligible score to `0` is unsafe: it could outrank eligible negative scores. We remove ineligible indexes before selection instead.

In [20]:
def vector_versions():
    return {
        "script": rank_cosine,
        "oop": lambda ids, matrix, user, k, eligible_mask=None:
            CosineRanker(ids, matrix).rank(user, k, eligible_mask),
    }


def test_vector_contract():
    for name, rank in vector_versions().items():
        result = rank(["A", "B", "C"], [[1, 0], [0, 1], [1, 1]], [1, 0], 2,
                      [False, True, True])
        assert [x[0] for x in result] == ["C", "B"]
        assert np.allclose([x[1] for x in result], [1 / math.sqrt(2), 0],
                           rtol=1e-10, atol=1e-12)

        # Catches using raw dot product in place of cosine similarity.
        magnitude = rank(["aligned", "large_diagonal"], [[1, 0], [10, 10]], [1, 0], 2)
        assert [x[0] for x in magnitude] == ["aligned", "large_diagonal"]

        # Exact score ties are resolved by ID, not original row order.
        ties = rank(["B", "A"], [[1, 0], [1, 0]], [1, 0], 2)
        assert ties == [("A", 1.0), ("B", 1.0)]

        # Explicit zero-vector convention + negative-score behavior.
        zero_query = rank(["B", "A"], [[1, 0], [0, 1]], [0, 0], 10)
        assert zero_query == [("A", 0.0), ("B", 0.0)]
        negative = rank(["blocked", "negative"], [[1, 0], [-1, 0]], [1, 0], 1,
                        [False, True])
        assert negative == [("negative", -1.0)]
        assert rank(["zero"], [[0, 0]], [1, 0], 1) == [("zero", 0.0)]
        assert rank(["A"], [[1, 0]], [1, 0], 1, [False]) == []
        assert rank(["A"], [[1, 0]], [1, 0], 0) == []
        assert rank([], np.empty((0, 2)), [1, 0], 3) == []
        print(name, "cosine ranker: norms, masks, ties, and zero-vector cases passed")


def test_vector_errors_and_cache():
    for name, rank in vector_versions().items():
        expect_raises(ValueError, lambda: rank(["A"], [[1, 0]], [1, 0, 0], 1))
        expect_raises(ValueError, lambda: rank(["A", "A"], [[1, 0], [0, 1]], [1, 0], 1))
        expect_raises(ValueError, lambda: rank(["A"], [[float("nan"), 0]], [1, 0], 1))
        expect_raises(ValueError, lambda: rank(["A"], [[1, 0]], [1, 0], 1, [1]))
        expect_raises(ValueError, lambda: rank(["A"], [[1, 0]], [1, 0], 1, [True, False]))
        expect_raises(ValueError, lambda: rank(["A"], [[1, 0]], [1, 0], -1))

    original = np.asarray([[1, 0], [0, 1]], dtype=float)
    cached = CosineRanker(["A", "B"], original)
    original[:] = 0  # Editing the input matrix must not change the normalized cache.
    assert cached.rank([1, 0], 1) == [("A", 1.0)]
    print("Vector error and cache-ownership checks passed.")


test_vector_contract()
test_vector_errors_and_cache()

script cosine ranker: norms, masks, ties, and zero-vector cases passed
oop cosine ranker: norms, masks, ties, and zero-vector cases passed
Vector error and cache-ownership checks passed.


### Talking points — cosine ranking

> “I will confirm that the item matrix is `(N,D)`, the user vector is `(D,)`, and IDs align with rows.”

> “Cosine is a dot product after normalization. `axis=1, keepdims=True` gives one column-shaped norm per item, so row-wise division broadcasts correctly.”

> “I will state the zero-vector policy explicitly. A similarity score, especially a negative one, is not directly a valid sampling weight.”

> “I will apply eligibility before top-K and use score descending, ID ascending for exact ties.”

> “The class caches normalized item vectors. It still spends `O(ND)` on full-catalog scoring per query, plus `O(M log M)` to sort M eligible items; stored vectors require `O(ND)` space.”

> “I will start with sorting. If K is small and selection dominates, a heap or partition can reduce selection work, but neither removes vector-scoring cost.”

**Bridge to the coupon problem:** eligibility can still come from expiry, minimum spend, and redemption history. Replace only the score source; preserve the item-ID alignment and tests.

<a id="testing-rehearsal"></a>
# 4. Testing and change-request rehearsal — the third prep area

This section is deliberately a cross-cutting skill, not an invented third interview prompt.

### 4A. Script-style test runner

The functions below contain ordinary `assert` statements. Calling them executes the real algorithms and raises immediately on a mismatch. No testing library is required for these checks.

After an edit, rerun the implementation cell, then this cell. If a class definition changed, recreate instances rather than assuming existing objects acquired the new definition.

In [21]:
# A single place to rerun the regression checks after changing implementation code.
REVISION_CHECKS = [
    test_sampler_contract,
    test_weight_update,
    test_coupon_contract,
    test_coupon_errors_and_input_safety,
    test_scorer_change,
    test_vector_contract,
    test_vector_errors_and_cache,
]

for check in REVISION_CHECKS:
    check()
print(f"\nAll {len(REVISION_CHECKS)} script-style check groups passed.")

script sampler: boundaries and invalid-input cases passed
oop sampler: boundaries and invalid-input cases passed
Weight-update checks passed.
script coupon recommender: ranking, eligibility, and cold-start cases passed
oop coupon recommender: ranking, eligibility, and cold-start cases passed
Coupon error and input-safety checks passed.
Scorer replacement preserved eligibility in both implementations.
script cosine ranker: norms, masks, ties, and zero-vector cases passed
oop cosine ranker: norms, masks, ties, and zero-vector cases passed
Vector error and cache-ownership checks passed.

All 7 script-style check groups passed.


### 4B. OOP-style tests with the standard-library `unittest`

A test class groups independently named examples. `self.assertEqual` compares exact results; `self.assertAlmostEqual` compares numeric results; `with self.assertRaises(...)` checks an error path. A runner discovers methods whose names start with `test_`. [R9]

**Notebook detail:** run a loaded suite instead of calling `unittest.main()`, which would parse the notebook kernel's command-line arguments. `verbosity=2` prints each test name. Prefer plain assertions during a short coding interview unless a test framework is requested.

In [22]:
class TestInterviewImplementations(unittest.TestCase):
    """Small examples showing the same contracts through a standard test class."""

    def test_integer_boundary(self):
        obj = WeightedSampler(["A", "B", "C", "D"], [1, 3, 0, 2],
                              rng=FixedRNG(integers=[4]))
        self.assertEqual(obj.sample(), "D")

    def test_function_and_object_match(self):
        functional = weighted_sample(["A", "B"], [1, 4], 20, random.Random(9))
        object_result = WeightedSampler(["A", "B"], [1, 4], random.Random(9)).sample_many(20)
        self.assertEqual(functional, object_result)

    def test_rejected_update_preserves_state(self):
        obj = WeightedSampler(["A", "B"], [1, 0], random.Random(1))
        with self.assertRaises(ValueError):
            obj.update_weight(0, 0)
        self.assertEqual(obj.sample_many(4), ["A"] * 4)

    def test_coupon_order_and_score(self):
        result = CouponRecommender(COUPONS).recommend(USER, HISTORY, 3, TODAY)
        self.assertEqual([item_id for item_id, _ in result], ["B", "F", "C"])
        self.assertAlmostEqual(result[0][1], 0.7 * (2 / 3) + 0.3 * 0.6)

    def test_cold_start(self):
        result = recommend_coupons({"id": "new", "cart_total": 800}, COUPONS, HISTORY, 1, TODAY)
        self.assertEqual(result, [("G", 1.0)])

    def test_new_scorer_keeps_filtering(self):
        result = CouponRecommender(COUPONS, scorer=popularity_only).recommend(USER, HISTORY, 10, TODAY)
        self.assertEqual([item_id for item_id, _ in result], ["G", "C", "B", "F"])

    def test_cosine_not_raw_dot_product(self):
        obj = CosineRanker(["aligned", "large_diagonal"], [[1, 0], [10, 10]])
        self.assertEqual(obj.rank([1, 0], 1), [("aligned", 1.0)])

    def test_ineligible_item_cannot_win(self):
        obj = CosineRanker(["A", "B", "C"], [[1, 0], [0, 1], [1, 1]])
        result = obj.rank([1, 0], 2, [False, True, True])
        self.assertEqual([item_id for item_id, _ in result], ["C", "B"])


suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestInterviewImplementations)
unit_test_result = unittest.TextTestRunner(stream=sys.stdout, verbosity=2).run(suite)
assert unit_test_result.wasSuccessful(), "Fix the failing test before continuing."

test_cold_start (__main__.TestInterviewImplementations.test_cold_start) ... 

ok


test_cosine_not_raw_dot_product (__main__.TestInterviewImplementations.test_cosine_not_raw_dot_product) ... 

ok


test_coupon_order_and_score (__main__.TestInterviewImplementations.test_coupon_order_and_score) ... 

...

test_new_scorer_keeps_filtering (__main__.TestInterviewImplementations.test_new_scorer_keeps_filtering) ... 

ok


test_rejected_update_preserves_state (__main__.TestInterviewImplementations.test_rejected_update_preserves_state) ... 

ok


----------------------------------------------------------------------
Ran 8 tests in 0.004s

OK


### Talking points — testing and adapting code

> “I will implement a correct baseline, then execute a small hand-checkable case before optimizing.”

> “I will test boundary behavior deterministically. For random code, I will separate the random input from the mapping logic.”

> “I will pass the clock and random generator explicitly so tests do not depend on today's date or unrelated global RNG calls.”

> “Before accepting a new requirement, I will restate what changes and identify which existing behavior must stay unchanged.”

> “After the change, I will rerun the previous tests as well as the new case. I will compare floating-point scores with a tolerance but keep IDs and rank order exact.”

**Two-minute rehearsal:** explain why the weight-update method validates before committing state, then explain why the new coupon scorer must not bypass eligibility. Those are small code changes with observable correctness conditions.

<a id="top-k-reference"></a>
# 5. Optional reference — sort, heap, and NumPy partition

**Use full sorting first.** The following comparison is for a follow-up about a large candidate set, not a request to memorize three separate solutions.

| Method | Selection cost for M candidates | Main advantage / caveat |
|---|---|---|
| Full sort | `O(M log M)` | Simple; explicit score and ID tie-break. |
| `heapq.nsmallest` with `(-score, id)` key | `O(M log K)` for small positive K | Retains only K winners in its heap; total pipeline memory still depends on what was materialized upstream. |
| `np.argpartition` + sort the selected K | Linear partition work + `O(K log K)` ordering | Selected subset is not already sorted; exact ties at the cutoff need a separate membership policy. |

`heapq.nsmallest(k, rows, key=...)` is convenient here because “smallest negative score” means “largest original score,” while IDs remain ascending. [R8]

`np.argpartition(a, kth)` returns **indexes**, and `kth` is a **zero-based order-statistic position**, not “how many results.” For the top K scores, partition `-scores` at `kth=K-1` and take the first K indexes. Then order those winners. [R14]

In [23]:
# Small demonstration with UNIQUE scores: no ambiguity at the selection cutoff.
ranked_rows_demo = [("A", 0.2), ("B", 0.9), ("C", 0.4), ("D", 0.6)]
k_demo = 2
ranking_key = lambda row: (-row[1], row[0])

by_sort = sorted(ranked_rows_demo, key=ranking_key)[:k_demo]
by_heap = heapq.nsmallest(k_demo, ranked_rows_demo, key=ranking_key)

score_array_demo = np.asarray([score for _, score in ranked_rows_demo])
effective_k = min(k_demo, len(ranked_rows_demo))
if effective_k == 0:
    by_partition = []
else:
    # Negative scores turn a largest-score request into smallest-value partitioning.
    chosen_indexes = np.argpartition(-score_array_demo, kth=effective_k - 1)[:effective_k]
    chosen_rows = [ranked_rows_demo[index] for index in chosen_indexes]
    by_partition = sorted(chosen_rows, key=ranking_key)

assert by_sort == by_heap == by_partition == [("B", 0.9), ("D", 0.6)]
print("sort:     ", by_sort)
print("heap:     ", by_heap)
print("partition:", by_partition)

sort:      [('B', 0.9), ('D', 0.6)]
heap:      [('B', 0.9), ('D', 0.6)]
partition: [('B', 0.9), ('D', 0.6)]


### Partition traps worth remembering

**The selected K are not necessarily sorted.** Partition only establishes membership around an order-statistic boundary.

**Sorting those K does not repair cutoff-tie membership.** Suppose several items share the Kth score and only some fit. Partition may select an arbitrary subset of the tied items before you sort. For the required ID tie-break, use full sorting or a tie-aware heap; a more elaborate partition solution must explicitly resolve all items tied at the cutoff.

**Handle `k=0` and `k>N`.** `kth=k-1` is only appropriate after confirming positive K and bounding it to N. Also remember that `[-0:]` means the entire sequence, not an empty result.

**The full algorithm may still be expensive.** A faster top-K step does not remove the cost of scoring all candidates or retaining all their vectors. [R8, R14]

## Last-pass comparison: why keep state in an object?

| Problem | Script-style state | Object state | Actual benefit of the class |
|---|---|---|---|
| Weighted sampler | Cumulative array built per function call | Items, weights, cumulative array, RNG | Reuse setup across separate draws; provide a controlled update method. |
| Coupon recommender | Catalog, user, history, policy passed to the function | Catalog snapshot and scoring policy | Reuse the catalog/policy; keep each user's context request-local. |
| Cosine ranker | Item normalization repeated per call | Normalized item vectors and aligned IDs | Avoid repeated normalization; still score the catalog per user. |
| Tests | Functions containing assertions | Named methods in a `TestCase` | Group and independently report behaviors; not required for a short interview. |

**Do not claim “OOP is faster” by itself.** The speed benefit comes from reusing previously computed state. A well-designed function-based program can also retain that state explicitly.

## Final recall — answer out loud before expanding

<details>
<summary><strong>Why bisect_right rather than bisect_left?</strong></summary>

Our intervals are left-inclusive and right-exclusive. A ticket equal to a cumulative endpoint belongs to the next nonempty interval. `bisect_right` moves past equal endpoints, including duplicate endpoints produced by zero weights.
</details>

<details>
<summary><strong>Why not take K coupons and then filter?</strong></summary>

Filtering could remove several of those K and leave unused eligible coupons below the cutoff. Apply eligibility before selecting K.
</details>

<details>
<summary><strong>What does key=lambda row: (-row[1], row[0]) do?</strong></summary>

For a `(coupon_id, score)` pair, it creates a sortable key `(-score, coupon_id)`. Ascending tuple sorting therefore yields score descending and ID ascending.
</details>

<details>
<summary><strong>What does axis=1, keepdims=True mean?</strong></summary>

Reduce each row across its feature coordinates while retaining a length-one feature axis. An `(N,D)` matrix produces `(N,1)` row norms, which broadcast across the D coordinates of each row.
</details>

<details>
<summary><strong>What must stay true when a scorer changes?</strong></summary>

Eligibility, uniqueness, item-ID alignment, finite score outputs, deterministic ties, and the K contract must stay intact. Only the ordering implied by the new score should change.
</details>

<details>
<summary><strong>What is the most useful next optimization?</strong></summary>

Name the measured or stated bottleneck first: repeated sampler setup, frequent weight updates, history scanning, item scoring, or top-K selection. Optimize that stage and rerun its correctness tests.
</details>

<a id="references"></a>
# 6. Official API references

The algorithms, fixtures, examples, and tests are original teaching material. These primary references support the language/library semantics explained above; checked on **11 September 2026**. You do not need to open them to use the notebook.

| Ref | Documentation | Used for |
|---|---|---|
| R1 | [Python `random`](https://docs.python.org/3/library/random.html) | RNG objects, integer/float draws, weighted choices, replacement semantics. |
| R2 | [Python `bisect`](https://docs.python.org/3/library/bisect.html) | Left/right insertion boundary semantics. |
| R3 | [Python sorting guide](https://docs.python.org/3/howto/sorting.html) | Key functions, stable sorting, in-place versus returned sorting. |
| R4 | [Python `collections.Counter`](https://docs.python.org/3/library/collections.html#collections.Counter) | Counting and dictionary-like operations. |
| R5 | [NumPy `linalg.norm`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.norm.html) | Norms, `axis`, and `keepdims`. |
| R6 | [NumPy `divide`](https://numpy.org/doc/stable/reference/generated/numpy.divide.html) | Broadcasting, `out`, and `where`. |
| R7 | [NumPy `matmul`](https://numpy.org/doc/stable/reference/generated/numpy.matmul.html) | Matrix-vector products and `@`. |
| R8 | [Python `heapq`](https://docs.python.org/3/library/heapq.html) | Heap-based top-K and `nsmallest`. |
| R9 | [Python `unittest`](https://docs.python.org/3/library/unittest.html) | Test cases, assertions, suites, and runners. |
| R10 | [Python `itertools.accumulate`](https://docs.python.org/3/library/itertools.html#itertools.accumulate) | Running accumulation. |
| R11 | [Python classes tutorial](https://docs.python.org/3/tutorial/classes.html) | Instances, attributes, methods, and `self`. |
| R12 | [NumPy `asarray`](https://numpy.org/doc/stable/reference/generated/numpy.asarray.html) | Array conversion, dtype, and copy behavior. |
| R13 | [NumPy `allclose`](https://numpy.org/doc/stable/reference/generated/numpy.allclose.html) | Float-comparison tolerances. |
| R14 | [NumPy `argpartition`](https://numpy.org/doc/stable/reference/generated/numpy.argpartition.html) | Indirect partitioning, kth position, and unordered partitions. |
| R15 | [Python `math`](https://docs.python.org/3/library/math.html) | `isfinite`, `isclose`, and `nextafter`. |

In [24]:
# Final status is produced by actual executed tests, not a hard-coded success claim.
assert unit_test_result.wasSuccessful()
print("Revision notebook complete.")
print(f"Script-style regression groups: {len(REVISION_CHECKS)} passed")
print(f"unittest cases: {unit_test_result.testsRun} passed")
print("Main implementations: weighted_sample / WeightedSampler")
print("                      recommend_coupons / CouponRecommender")
print("Vector extension:     rank_cosine / CosineRanker")
print("Next revision: reproduce the two main algorithms without looking.")

Revision notebook complete.
Script-style regression groups: 7 passed
unittest cases: 8 passed
Main implementations: weighted_sample / WeightedSampler
                      recommend_coupons / CouponRecommender
Vector extension:     rank_cosine / CosineRanker
Next revision: reproduce the two main algorithms without looking.
